# SASHIMI-SI: independent scientific comparison

A is corrected upstream `e17d3664dac677b604fd4ff02fb2af105a6937fa`. B adds an independent 50-digit analytic NFW inverse, evaluates only the selected cross-section branch, and computes the effective cross section at 65-digit precision. The product evaluates the same defining expression as a positive integral to avoid cancellation. The product uses a separate root solver and the common executor. This notebook checks migration agreement, not grid convergence or SIDM calibration. The total-cross-section formula is under a separate scientific review; it is not used by this catalog calculation.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from sashimi_si import SubhaloProperties
reference_dir = Path("tests/references")
if not reference_dir.exists():
    reference_dir = Path("../tests/references")
provenance = json.loads((reference_dir/"B-accurate.json").read_text())
parameters = provenance["calculation"]["parameters"]
model = SubhaloProperties()
result = model.subhalo_properties_calc(**parameters)
catalogs = model.subhalo_catalogs_calc(**parameters)
reference = np.load(reference_dir/"B-accurate.npz")
metrics = []
for i, actual in enumerate(result):
    expected = reference[f"tuple_{i}"]
    if i in (25, 26):
        np.testing.assert_array_equal(actual, expected)
    else:
        np.testing.assert_allclose(actual, expected, rtol=5e-12, atol=1e-300)
    metrics.append(float(np.max(np.abs(np.asarray(actual,float)-expected)/np.maximum(np.abs(expected),1e-300))))
print("All 27 fields agree. Maximum relative difference:", max(metrics))
print("Reference source:", provenance["source_revision"])
print("Product specification:", catalogs["sidm"].metadata["calculation_specification"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
edges = np.geomspace(1e4, 1e7, 10)
for state, index in [("cdm_reference", 23), ("sidm", 24)]:
    catalog = catalogs[state]
    counts, _ = np.histogram(catalog.columns["m_bound"], edges, weights=catalog.weight_final)
    ref_counts, _ = np.histogram(reference["tuple_11"], edges, weights=reference[f"tuple_{index}"])
    np.testing.assert_allclose(counts, ref_counts, rtol=5e-12)
    axes[0].step(np.sqrt(edges[:-1]*edges[1:]), counts/np.diff(np.log(edges)), where="mid", label=state)
axes[0].set(xscale="log", xlabel="Bound mass [Msun]", ylabel="dN/dln M")
axes[0].legend()
axes[1].semilogy(np.arange(27), np.maximum(metrics,1e-16), "o")
axes[1].set(xlabel="Historical tuple field", ylabel="Max relative B/C difference")
fig.tight_layout()
plt.show()

In [ ]:
formation_dir = reference_dir.parent/"formation_reference"
formation_provenance = json.loads((formation_dir/"B-accurate-formation.json").read_text())
formed = model.subhalo_catalogs_calc(**formation_provenance["calculation"]["parameters"])
valid = formed["sidm"].columns["valid_accretion"]
assert valid.size == 168 and valid.sum() == 164
for state in formed.values():
    assert np.all(state.weight_final[~valid] == 0)
    assert all(np.all(np.isfinite(v)) for v in state.columns.values())
print("Formation boundary: 164 physical nodes, 4 unformed nodes with explicit validity flags.")

Both state masks, their independent base weights and representative mass functions are checked. Historical inverse interpolation error is not included in the B/C tolerance. Host mass, redshift, both quadrature orders, solver, threshold and SI parameters are fixed. Larger-grid convergence and strong-interaction calibration remain separate release evidence.